# 05 · App scorecard and recommendations

**Author:** Mohamed Elsadany · data collected from Google Play on 28 Oct 2026

**My question:** which app gives the best experience, and what is the single biggest fix for each one?

In this notebook I bring together everything from notebooks 02–04 into one scorecard per app, look at what changed
in the last six months, write my recommendations, and export the tables my Power BI dashboard is built on
(`dashboard/data/`). All numbers use each app's full-coverage period, like the other notebooks.

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from topics import TOPICS, TOPIC_LABELS, add_topics                          # noqa: E402
from ratings import weekly_counts, find_surges                               # noqa: E402

warnings.filterwarnings("ignore", message="obj.round has no effect")
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)
APPS = ["InstaPay", "Fawry", "Telda", "valU", "Vodafone Cash"]

df = pd.read_csv(ROOT / "data" / "processed" / "reviews_clean.csv", parse_dates=["at", "coverage_start"])
df["content_clean"] = df["content_clean"].fillna("")
sent = pd.read_csv(ROOT / "data" / "processed" / "reviews_sentiment.csv")[["reviewId", "model_sentiment", "model_score"]]
df = df.merge(sent, on="reviewId", how="left")
meta = pd.read_csv(ROOT / "data" / "raw" / "apps_metadata.csv").set_index("app")

# Topics for every review with 3+ words (the scorecard uses the negative ones).
TOPIC_COLS = [f"topic_{k}" for k in TOPICS]
tx = add_topics(df.loc[df["text_usable"], ["content_clean"]])
df = df.join(tx[TOPIC_COLS + ["n_topics"]])
df[TOPIC_COLS] = df[TOPIC_COLS].fillna(False).astype(bool)
df["n_topics"] = df["n_topics"].fillna(0).astype(int)
for col in ["is_negative", "is_positive", "text_usable", "in_full_coverage", "has_reply"]:
    df[col] = df[col].astype(bool)

# Hidden pain point = 4-5★ review with 3+ words whose text the model is confident is negative (notebook 04).
df["hidden_pain_base"] = df["is_positive"] & df["text_usable"] & df["lang_detected"].isin(["arabic", "english"])
df["hidden_pain_point"] = df["hidden_pain_base"] & (df["model_sentiment"] == "negative") & (df["model_score"] >= 0.8)

cov = df[df["in_full_coverage"]].copy()
END = cov["at"].max()
print(f"{len(cov):,} reviews in the full-coverage period, latest review {END:%d %b %Y}")

57,231 reviews in the full-coverage period, latest review 27 Oct 2026


## 1. The scorecard

One row per app. `avg_last_6m` and `avg_prev_6m` compare the last six months with the six before, so the trend shows
whether things are getting better or worse. Vodafone Cash has only six months of full coverage, so it has no trend.

In [2]:
neg_text = cov[cov["is_negative"] & cov["text_usable"]]
topic_share = neg_text.groupby("app")[TOPIC_COLS].mean().reindex(APPS)
topic_share.columns = [TOPIC_LABELS[c.replace("topic_", "")] for c in topic_share.columns]

last = cov[cov["at"] > END - pd.Timedelta(days=182)]
prev = cov[(cov["at"] <= END - pd.Timedelta(days=182)) & (cov["at"] > END - pd.Timedelta(days=364))]
neg = cov[cov["is_negative"]]

score = pd.DataFrame({
    "reviews": cov.groupby("app").size(),
    "store_rating": meta["score"].round(2),
    "avg_stars": cov.groupby("app")["score"].mean(),
    "pct_1_star": cov.groupby("app")["score"].apply(lambda s: (s == 1).mean() * 100),
    "avg_last_6m": last.groupby("app")["score"].mean(),
    "avg_prev_6m": prev.groupby("app")["score"].mean().where(prev.groupby("app").size() >= 100),
    "top_complaint": topic_share.idxmax(axis=1),
    "top_complaint_pct": topic_share.max(axis=1) * 100,
    "hidden_pain_pct": cov.groupby("app").apply(
        lambda g: g["hidden_pain_point"].sum() / g["hidden_pain_base"].sum() * 100, include_groups=False),
    "replied_to_negative_pct": neg.groupby("app")["has_reply"].mean() * 100,
    "median_reply_hours": neg.groupby("app")["reply_hours"].median(),
}).reindex(APPS)
score["trend"] = (score["avg_last_6m"] - score["avg_prev_6m"]).round(2)
score = score.round({"avg_stars": 2, "pct_1_star": 1, "avg_last_6m": 2, "avg_prev_6m": 2,
                     "top_complaint_pct": 1, "hidden_pain_pct": 1, "replied_to_negative_pct": 1,
                     "median_reply_hours": 1})
score.sort_values("avg_stars", ascending=False)

,reviews,store_rating,avg_stars,pct_1_star,avg_last_6m,avg_prev_6m,top_complaint,top_complaint_pct,hidden_pain_pct,replied_to_negative_pct,median_reply_hours,trend
app,,,,,,,,,,,,
Telda,8377,4.75,3.92,22.4,3.72,4.09,"Card delivery, branches or ATM use",24.6,3.4,18.6,19.1,-0.37
InstaPay,17637,4.29,3.62,30.1,3.55,3.44,"App won't open, crashes or is slow",51.3,8.6,73.4,72.7,0.11
Fawry,19354,3.85,3.28,37.2,2.84,3.08,Customer service unreachable or unhelpful,34.4,6.7,98.6,4.8,-0.25
valU,11148,3.49,2.90,48.4,2.09,2.80,Customer service unreachable or unhelpful,22.3,6.1,87.5,1106.5,-0.72
Vodafone Cash,715,4.38,1.92,68.5,1.92,NaN,"App won't open, crashes or is slow",47.4,21.6,94.8,0.7,NaN


## 2. Which app gives the best experience?

I rank the apps on the two measures that matter most to a user: how often reviews are 1★, and the average rating.
I don't build a combined score, because the weights would be my own opinion; the scorecard above shows everything.

In [3]:
rank = score[["avg_stars", "pct_1_star", "trend"]].copy()
rank["rank_by_avg_stars"] = rank["avg_stars"].rank(ascending=False).astype(int)
rank["rank_by_1_star"] = rank["pct_1_star"].rank().astype(int)
rank.sort_values("rank_by_avg_stars")

,avg_stars,pct_1_star,trend,rank_by_avg_stars,rank_by_1_star
app,,,,,
Telda,3.92,22.4,-0.37,1,1
InstaPay,3.62,30.1,0.11,2,2
Fawry,3.28,37.2,-0.25,3,3
valU,2.90,48.4,-0.72,4,4
Vodafone Cash,1.92,68.5,NaN,5,5


## 3. What changed in the last six months?

For each app, the change in each topic's share of negative reviews (last six months minus the six before),
showing the biggest moves only. Vodafone Cash is left out because it has no earlier period.

In [4]:
def topic_mix(frame):
    g = frame[frame["is_negative"] & frame["text_usable"]]
    out = g.groupby("app")[TOPIC_COLS].mean() * 100
    out.columns = [TOPIC_LABELS[c.replace("topic_", "")] for c in out.columns]
    return out

change = (topic_mix(last) - topic_mix(prev)).reindex(["InstaPay", "Fawry", "Telda", "valU"])
counts = pd.DataFrame({"neg_reviews_last_6m": last[last["is_negative"] & last["text_usable"]].groupby("app").size(),
                       "neg_reviews_prev_6m": prev[prev["is_negative"] & prev["text_usable"]].groupby("app").size()})
display(counts.reindex(change.index))
moves = (change.stack().rename("change_pts").reset_index()
               .rename(columns={"level_1": "topic"})
               .assign(abs=lambda x: x["change_pts"].abs())
               .sort_values("abs", ascending=False)
               .groupby("app").head(3).drop(columns="abs").round(1))
moves.sort_values(["app", "change_pts"])

,neg_reviews_last_6m,neg_reviews_prev_6m
app,,
InstaPay,1221,2460.0
Fawry,1250,997.0
Telda,242,106.0
valU,254,145.0


,app,topic,change_pts
15,Fawry,Customer service unreachable or unhelpful,-11.4
13,Fawry,"Money deducted, transfer failed or stuck",-4.4
12,Fawry,"App won't open, crashes or is slow",13.2
0,InstaPay,"App won't open, crashes or is slow",-12.5
7,InstaPay,Can't use it from abroad,-1.5
2,InstaPay,"Sign-up, login, verification or waiting list",4.7
29,Telda,"Card delivery, branches or ATM use",-7.5
33,Telda,Accuses the company of fraud or theft,3.1
27,Telda,Customer service unreachable or unhelpful,4.3
39,valU,Customer service unreachable or unhelpful,-7.8


## 4. My recommendations

Two or three per app, each tied to a number from this project. They are based only on public reviews: the companies'
own data (crash logs, call-center volumes) would be needed to size the work, so I state them as priorities to
investigate, not as finished solutions.

In [5]:
RECS = [
    ("InstaPay", 1, "Release safety: roll updates out to a small share of users first and stop the rollout when crash reports jump.",
     "Version 1.12.0 (Feb 2026) brought ~1,200 one-star reviews in two weeks (~26x usual); 'app won't open' is in 51% of negative reviews."),
    ("InstaPay", 2, "Make sign-up work without SMS problems: an in-app OTP option, clearer errors, and support for users abroad.",
     "Sign-up / SMS verification is in 18% of negative reviews; 'not working from abroad' in 3%."),
    ("InstaPay", 3, "When the security check blocks a phone, say which app or setting caused it.",
     "Security-block and unsupported-phone complaints ('this app is hooked') are in 3% of negative reviews, and users say they can't tell what to remove."),
    ("Fawry", 1, "Stabilize the app before new features: test updates on more phone models and stage rollouts.",
     "'App won't open / crashes' rose from 26% to 39% of negative reviews in the last six months; version 6.9.0 had 65% one-star reviews."),
    ("Fawry", 2, "Show the status of every failed payment or ATM withdrawal in the app, with an automatic refund date.",
     "Failed or stuck transactions are in 10% of Fawry's negative reviews (bill payments, recharges, Fawry card at ATMs), the highest of all five apps."),
    ("Fawry", 3, "Keep investing in customer service: it is improving but still the top complaint.",
     "Customer service is in 34% of negative reviews overall, down from 29% to 17% in the last six months compared with the six before."),
    ("Telda", 1, "Fix the card journey: delivery or pickup outside Cairo, delivery tracking, and clear card fees.",
     "The card is the top complaint (25% of negative reviews): waiting time, branches, and the 99 EGP card fee."),
    ("Telda", 2, "Answer negative reviews and support chats faster.",
     "Telda replies to only 19% of negative reviews, and customer-service complaints rose from 21% to 26% in the last six months."),
    ("Telda", 3, "Investigate why ratings are slipping before it becomes a trend.",
     "Average rating fell from 4.09 to 3.72 and the 1-star share rose from 17% to 26% (last six months vs the six before)."),
    ("valU", 1, "Tell applicants the reason for a rejection, and pre-check eligibility in the app before they travel to a branch.",
     "Rejected applications or low limits are in 12% of negative reviews, often 'rejected after hours at a branch with no reason'."),
    ("valU", 2, "Set a reply target for complaints.",
     "valU replies to 88% of negative reviews, but the median reply takes 46 days."),
    ("valU", 3, "Stop the decline: fix app stability and make fees and interest clear before purchase.",
     "Average rating fell from 2.80 to 2.09 in the last six months; 'app won't open' rose from 10% to 23% of negative reviews; fees and interest 7%, fraud accusations 9%."),
    ("Vodafone Cash", 1, "Make the wallet page inside Ana Vodafone fast and reliable (loading, lag and crashes).",
     "'App won't open / crashes / slow' is in 47% of negative wallet reviews; 22% of 4-5★ wallet reviews still describe a problem."),
    ("Vodafone Cash", 2, "Review the wallet sign-up and verification steps.",
     "Sign-up, login or verification problems are in 14% of negative wallet reviews. (Only six months and ~700 reviews of full coverage.)"),
]
recs = pd.DataFrame(RECS, columns=["app", "priority", "recommendation", "evidence"])
recs

,app,priority,recommendation,evidence
0,InstaPay,1,Release safety: roll updates out to a small share of users first and stop the rollout when crash reports jump.,"Version 1.12.0 (Feb 2026) brought ~1,200 one-star reviews in two weeks (~26x usual); 'app won't open' is in 51% of negative reviews."
1,InstaPay,2,"Make sign-up work without SMS problems: an in-app OTP option, clearer errors, and support for users abroad.",Sign-up / SMS verification is in 18% of negative reviews; 'not working from abroad' in 3%.
2,InstaPay,3,"When the security check blocks a phone, say which app or setting caused it.","Security-block and unsupported-phone complaints ('this app is hooked') are in 3% of negative reviews, and users say they can't tell what to remove."
3,Fawry,1,Stabilize the app before new features: test updates on more phone models and stage rollouts.,'App won't open / crashes' rose from 26% to 39% of negative reviews in the last six months; version 6.9.0 had 65% one-star reviews.
4,Fawry,2,"Show the status of every failed payment or ATM withdrawal in the app, with an automatic refund date.","Failed or stuck transactions are in 10% of Fawry's negative reviews (bill payments, recharges, Fawry card at ATMs), the highest of all five apps."
5,Fawry,3,Keep investing in customer service: it is improving but still the top complaint.,"Customer service is in 34% of negative reviews overall, down from 29% to 17% in the last six months compared with the six before."
6,Telda,1,"Fix the card journey: delivery or pickup outside Cairo, delivery tracking, and clear card fees.","The card is the top complaint (25% of negative reviews): waiting time, branches, and the 99 EGP card fee."
7,Telda,2,Answer negative reviews and support chats faster.,"Telda replies to only 19% of negative reviews, and customer-service complaints rose from 21% to 26% in the last six months."
8,Telda,3,Investigate why ratings are slipping before it becomes a trend.,Average rating fell from 4.09 to 3.72 and the 1-star share rose from 17% to 26% (last six months vs the six before).
9,valU,1,"Tell applicants the reason for a rejection, and pre-check eligibility in the app before they travel to a branch.","Rejected applications or low limits are in 12% of negative reviews, often 'rejected after hours at a branch with no reason'."


## 5. Exporting the tables for Power BI

The dashboard reads plain CSV files from `dashboard/data/`. I use a small star schema:

| Table | One row per | Used for |
|---|---|---|
| `fact_reviews.csv` | review | every measure; 0/1 flags so DAX can use `SUM` |
| `review_topics.csv` | review × topic | topic shares (a review can have several topics) |
| `dim_topic.csv` | topic | topic names and sort order |
| `dim_app.csv` | app | store rating, installs, coverage start, app color |
| `surge_events.csv` | complaint surge | the events table on the Trends page |
| `recommendations.csv` | recommendation | the Recommendations page |

In [6]:
OUT = ROOT / "dashboard" / "data"
OUT.mkdir(parents=True, exist_ok=True)

flag = lambda s: s.fillna(False).astype(bool).astype(int)
fact = pd.DataFrame({
    "review_id": df["reviewId"],
    "app": df["app"],
    "review_date": df["at"].dt.date,
    "stars": df["score"].astype(int),
    "star_sentiment": df["star_sentiment"],
    "is_one_star": flag(df["score"] == 1),
    "is_negative": flag(df["is_negative"]),
    "is_positive": flag(df["is_positive"]),
    "language": df["lang_detected"],
    "word_count": df["word_count"].astype(int),
    "text_usable": flag(df["text_usable"]),
    "in_full_coverage": flag(df["in_full_coverage"]),
    "has_reply": flag(df["has_reply"]),
    "reply_hours": df["reply_hours"],
    "app_version": df["appVersion"],
    "model_sentiment": df["model_sentiment"],
    "model_score": df["model_score"].round(3),
    "hidden_pain_base": flag(df["hidden_pain_base"]),
    "hidden_pain_point": flag(df["hidden_pain_point"]),
    "n_topics": df["n_topics"],
    "review_text": df["content"].fillna("").str.replace(r"\s+", " ", regex=True).str[:300],
})
fact.to_csv(OUT / "fact_reviews.csv", index=False, encoding="utf-8-sig")

bridge = (df.loc[df["text_usable"], ["reviewId"] + TOPIC_COLS]
            .melt(id_vars="reviewId", var_name="topic_id", value_name="has")
            .query("has").drop(columns="has")
            .assign(topic_id=lambda x: x["topic_id"].str.replace("topic_", "", regex=False))
            .rename(columns={"reviewId": "review_id"}))
bridge.to_csv(OUT / "review_topics.csv", index=False, encoding="utf-8-sig")

order = list(topic_share.mean().sort_values(ascending=False).index)
dim_topic = pd.DataFrame({"topic_id": list(TOPICS), "topic": [TOPIC_LABELS[k] for k in TOPICS]})
dim_topic["sort_order"] = dim_topic["topic"].map({t: i + 1 for i, t in enumerate(order)})
dim_topic.sort_values("sort_order").to_csv(OUT / "dim_topic.csv", index=False, encoding="utf-8-sig")

COLORS = {"InstaPay": "#2A78D6", "Fawry": "#EB6834", "Telda": "#1BAF7A", "valU": "#EDA100", "Vodafone Cash": "#E87BA4"}
dim_app = pd.DataFrame({
    "app": APPS,
    "store_name": meta.loc[APPS, "title"].values,
    "store_rating": meta.loc[APPS, "score"].round(2).values,
    "store_ratings_count": meta.loc[APPS, "ratings"].values,
    "installs": meta.loc[APPS, "installs"].values,
    "coverage_start": cov.groupby("app")["coverage_start"].first().reindex(APPS).dt.date.values,
    "color": [COLORS[a] for a in APPS],
    "sort_order": range(1, len(APPS) + 1),
})
dim_app.to_csv(OUT / "dim_app.csv", index=False, encoding="utf-8-sig")

# Complaint surges, same rules as notebook 02.
weekly = weekly_counts(cov)
surges = find_surges(weekly, "one_star", min_count=40)
surges = surges[surges["avg_stars"] < surges["app"].map(cov.groupby("app")["score"].mean())]
CAUSES = {
    ("InstaPay", "2026-02-23"): ("1.12.0", "App stopped opening after the 1.12.0 update"),
    ("Fawry", "2024-12-16"): ("5.0.46", "Fawry card: cash stuck at ATMs, customer service not answering"),
    ("Fawry", "2026-06-29"): ("6.9.0", "App crashing or not opening after the 6.9.0 update"),
    ("valU", "2022-02-28"): ("6.0.3", "Backlash at a promotion: 'fake offers', hidden fees"),
    ("valU", "2024-03-11"): ("8.0.3", "8.0.x update broke the app; logins blocked with developer options on"),
    ("Telda", "2022-10-17"): ("2.3.0", "Sign-up failing with a false 'check your internet' error"),
}
events = pd.DataFrame({
    "app": surges["app"],
    "start_date": surges["start"].dt.date,
    "end_date": surges["end"].dt.date,
    "one_star_reviews": surges["count"].astype(int),
    "times_usual": surges["peak_times_usual"].round(1),
    "app_version": [CAUSES.get((a, f"{s:%Y-%m-%d}"), ("", ""))[0] for a, s in zip(surges["app"], surges["start"])],
    "cause": [CAUSES.get((a, f"{s:%Y-%m-%d}"), ("", "not labeled"))[1] for a, s in zip(surges["app"], surges["start"])],
}).sort_values("one_star_reviews", ascending=False)
events.to_csv(OUT / "surge_events.csv", index=False, encoding="utf-8-sig")

recs.to_csv(OUT / "recommendations.csv", index=False, encoding="utf-8-sig")
score.to_csv(OUT / "app_scorecard.csv", encoding="utf-8-sig")

for f in sorted(OUT.glob("*.csv")):
    print(f"{f.name:24s} {f.stat().st_size / 1e6:6.1f} MB  {sum(1 for _ in open(f, encoding='utf-8-sig')) - 1:>7,} rows")

app_scorecard.csv           0.0 MB        5 rows
dim_app.csv                 0.0 MB        5 rows
dim_topic.csv               0.0 MB       12 rows
fact_reviews.csv           14.0 MB   71,749 rows
recommendations.csv         0.0 MB       14 rows
review_topics.csv           1.6 MB   30,257 rows
surge_events.csv            0.0 MB        6 rows


## 6. Numbers to check the dashboard against

After building the dashboard I compare a few of its numbers with these. With the page filter
`in_full_coverage = 1`, the Power BI cards and tables should show exactly the same values.

In [7]:
check = pd.DataFrame({
    "Reviews": cov.groupby("app").size().astype(int),
    "Avg Stars": cov.groupby("app")["score"].mean().round(2),
    "% 1-Star": (cov.groupby("app")["score"].apply(lambda s: (s == 1).mean()) * 100).round(1),
    "Hidden Pain %": score["hidden_pain_pct"],
    "Reply Rate (negative) %": score["replied_to_negative_pct"],
    "App not working % of negative": (topic_share["App won't open, crashes or is slow"] * 100).round(1),
}).reindex(APPS)
check.loc["All apps"] = [len(cov), round(cov["score"].mean(), 2), round((cov["score"] == 1).mean() * 100, 1),
                         round(cov["hidden_pain_point"].sum() / cov["hidden_pain_base"].sum() * 100, 1),
                         round(neg["has_reply"].mean() * 100, 1),
                         round(neg_text["topic_app_not_working"].mean() * 100, 1)]
check

,Reviews,Avg Stars,% 1-Star,Hidden Pain %,Reply Rate (negative) %,App not working % of negative
app,,,,,,
InstaPay,17637.0,3.62,30.1,8.6,73.4,51.3
Fawry,19354.0,3.28,37.2,6.7,98.6,22.7
Telda,8377.0,3.92,22.4,3.4,18.6,11.3
valU,11148.0,2.90,48.4,6.1,87.5,15.3
Vodafone Cash,715.0,1.92,68.5,21.6,94.8,47.4
All apps,57231.0,3.39,35.4,6.6,81.4,28.2


## What I found

1. **Telda gives the best experience** on every measure: the highest average rating (3.92★) and the lowest 1★ share
   (22%). But it is slipping: its average fell from 4.09 to 3.72 in the last six months.
2. **Vodafone Cash wallet reviews are the most negative** (69% one-star), and **valU is declining fastest**
   (average 2.80 → 2.09 in the last six months).
3. **InstaPay is the only app improving** (3.44 → 3.55), after recovering from its February 2026 release.
4. **The biggest fix for each app:** InstaPay and Vodafone Cash = app stability; Fawry = stability and failed-payment
   status; Telda = the card journey; valU = explaining rejections and replying in days, not weeks.

**Dashboard:** the Power BI report is built from `dashboard/data/` (see `dashboard/README.md`).